# EconBalancer 🪙 — A Macroeconomics Equilibrium Game

EconBalancer is a small educational game where the player acts as an economic
decision-maker trying to keep an economy **in equilibrium** — that is, keeping
**Aggregate Supply (AS)** and **Aggregate Demand (AD)** close to each other while
random economic *shocks* (consumer confidence swings, tech booms, spending cuts,
natural disasters) keep pushing them apart.

This notebook is a cleaned-up, refactored version of an early prototype. It:

* replaces tangled global state with a single, testable `EconBalancer` engine class,
* makes the game **actually winnable** by judging equilibrium with a tolerance band
  instead of exact equality,
* gives the three **roles** (Economist / Policy Maker / Business Leader) real,
  distinct gameplay effects,
* puts the **budget points** and **technology tokens** to real use,
* adds **scoring** and a **matplotlib visualization** of how the economy evolves, and
* keeps a fully **interactive ipywidgets UI** for playing inside Jupyter.

> The original four-stage prototype is kept alongside this file as
> `POME_EL_original.ipynb` for comparison.


## 1. Game design

**Goal.** Survive as many rounds as possible while keeping the economy balanced.

**Each round**
1. A random **shock** hits the economy, shifting AS and/or AD.
2. The player spends **budget points** on a **policy lever** to push AS or AD
   back toward balance.
3. Optionally spend a **technology token** to *double* the effect of that lever.
4. The economy is **balanced** when `|AS − AD| ≤ tolerance`. Stay balanced to score.

**Roles** change which levers are strong for you:

| Role            | Supply lever | Demand lever | Flavor                          |
|-----------------|:------------:|:------------:|---------------------------------|
| Economist       |   medium     |   medium     | balanced, flexible              |
| Policy Maker    |   weak       |   strong     | fiscal/monetary, demand-side    |
| Business Leader |   strong     |   weak       | production/investment, supply-side |

Because the engine has no hidden globals, every rule above is just a number you
can tweak in one place — which also makes the logic easy to unit-test.


In [1]:
import random
from dataclasses import dataclass, field


# ---- Configuration ---------------------------------------------------------

ROLES = {
    # role name : (supply_power, demand_power)
    "Economist":       (10, 10),
    "Policy Maker":    (6, 16),
    "Business Leader": (16, 6),
}

SCENARIOS = [
    # name,                              d_supply, d_demand
    ("Increase in consumer confidence",   0,        +12),
    ("Technological advancement",        +12,        0),
    ("Government spending cuts",           0,       -12),
    ("Natural disaster",                  -12,       -4),
    ("Export boom",                       +6,       +10),
    ("Credit crunch",                      -6,       -12),
]


@dataclass
class EconBalancer:
    """A small, self-contained engine for the EconBalancer game.

    All game state lives on the instance (no globals), so the same class can be
    driven by a console loop, an ipywidgets UI, or an automated test.
    """
    role: str = "Economist"
    tolerance: int = 5          # |AS - AD| within this counts as balanced
    budget_points: int = 60
    technology_tokens: int = 3
    action_cost: int = 10
    seed: int | None = None

    AS: int = 100
    AD: int = 100
    round_no: int = 0
    score: int = 0
    current_scenario: tuple | None = None
    history: list = field(default_factory=list)   # (round, AS, AD, gap)

    def __post_init__(self):
        self._rng = random.Random(self.seed)
        if self.role not in ROLES:
            raise ValueError(f"Unknown role {self.role!r}. Choose from {list(ROLES)}.")
        self._log()

    # ---- derived state ----
    @property
    def gap(self) -> int:
        return self.AS - self.AD

    @property
    def is_balanced(self) -> bool:
        return abs(self.gap) <= self.tolerance

    def status(self) -> str:
        if self.is_balanced:
            return "Economy is in equilibrium! ✅"
        return "Surplus (AS too high)." if self.gap > 0 else "Deficit (AD too high)."

    def can_act(self) -> bool:
        return self.budget_points >= self.action_cost

    # ---- game flow ----
    def new_round(self) -> tuple:
        """Advance to a new round and apply a random economic shock."""
        self.round_no += 1
        self.current_scenario = self._rng.choice(SCENARIOS)
        _, d_supply, d_demand = self.current_scenario
        self.AS += d_supply
        self.AD += d_demand
        self._log()
        return self.current_scenario

    def apply_policy(self, lever: str, direction: int, use_tech: bool = False) -> str:
        """Spend budget to nudge a lever.

        lever     : "supply" or "demand"
        direction : +1 to raise, -1 to lower
        use_tech  : spend one technology token to double the effect
        """
        if not self.can_act():
            return "Not enough budget points!"
        if direction not in (-1, 1):
            raise ValueError("direction must be +1 or -1")

        supply_power, demand_power = ROLES[self.role]
        power = supply_power if lever == "supply" else demand_power

        if use_tech and self.technology_tokens > 0:
            power *= 2
            self.technology_tokens -= 1

        delta = direction * power
        if lever == "supply":
            self.AS += delta
        elif lever == "demand":
            self.AD += delta
        else:
            raise ValueError("lever must be 'supply' or 'demand'")

        self.budget_points -= self.action_cost
        if self.is_balanced:
            self.score += 1
        self._log()
        return self.status()

    def _log(self):
        self.history.append((self.round_no, self.AS, self.AD, self.gap))

    def summary(self) -> str:
        return (f"Role: {self.role} | Round: {self.round_no} | "
                f"AS: {self.AS} | AD: {self.AD} | gap: {self.gap} | "
                f"Budget: {self.budget_points} | Tech: {self.technology_tokens} | "
                f"Score: {self.score}")


## 2. A scripted playthrough

The engine has no `input()` calls baked in, so we can drive it programmatically.
Below, a simple "auto-player" always pushes the lever that reduces the current
gap, spending a tech token whenever the gap is large. We seed the RNG so the run
is reproducible.

In [2]:
def auto_policy(game: "EconBalancer"):
    """A naive strategy: push whichever side reduces the gap."""
    gap = game.gap
    if gap > 0:          # surplus -> lower supply or raise demand
        lever, direction = ("demand", +1)
    else:                # deficit -> raise supply or lower demand
        lever, direction = ("supply", +1)
    use_tech = abs(gap) > 12
    return game.apply_policy(lever, direction, use_tech=use_tech)


game = EconBalancer(role="Economist", seed=7)
print("Welcome to EconBalancer!\n")

for _ in range(6):
    name, ds, dd = game.new_round()
    print(f"Round {game.round_no}: shock -> {name} (AS{ds:+}, AD{dd:+})")
    if not game.can_act():
        print("  Out of budget — the economy is on its own now.")
        break
    result = auto_policy(game)
    print(f"  Policy applied. {result}")
    print(f"  {game.summary()}\n")

print(f"Final score (rounds spent in equilibrium): {game.score}")


Welcome to EconBalancer!

Round 1: shock -> Government spending cuts (AS+0, AD-12)
  Policy applied. Economy is in equilibrium! ✅
  Role: Economist | Round: 1 | AS: 100 | AD: 98 | gap: 2 | Budget: 50 | Tech: 3 | Score: 1

Round 2: shock -> Technological advancement (AS+12, AD+0)
  Policy applied. Deficit (AD too high).
  Role: Economist | Round: 2 | AS: 112 | AD: 118 | gap: -6 | Budget: 40 | Tech: 2 | Score: 1

Round 3: shock -> Natural disaster (AS-12, AD-4)
  Policy applied. Surplus (AS too high).
  Role: Economist | Round: 3 | AS: 120 | AD: 114 | gap: 6 | Budget: 30 | Tech: 1 | Score: 1

Round 4: shock -> Credit crunch (AS-6, AD-12)
  Policy applied. Economy is in equilibrium! ✅
  Role: Economist | Round: 4 | AS: 114 | AD: 112 | gap: 2 | Budget: 20 | Tech: 1 | Score: 2

Round 5: shock -> Increase in consumer confidence (AS+0, AD+12)
  Policy applied. Economy is in equilibrium! ✅
  Role: Economist | Round: 5 | AS: 124 | AD: 124 | gap: 0 | Budget: 10 | Tech: 1 | Score: 3

Round 6: sho

## 3. Visualizing the economy

A picture makes the supply/demand tug-of-war much clearer than the printout.
The shaded band is the equilibrium tolerance zone around Aggregate Supply.

In [3]:
import matplotlib.pyplot as plt

rounds = [h[0] for h in game.history]
AS_vals = [h[1] for h in game.history]
AD_vals = [h[2] for h in game.history]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

ax1.plot(rounds, AS_vals, "o-", label="Aggregate Supply", color="#2a9d8f")
ax1.plot(rounds, AD_vals, "s-", label="Aggregate Demand", color="#e76f51")
ax1.fill_between(rounds,
                 [a - game.tolerance for a in AS_vals],
                 [a + game.tolerance for a in AS_vals],
                 color="#2a9d8f", alpha=0.12, label="Equilibrium band")
ax1.set_ylabel("Level")
ax1.set_title("EconBalancer — economy over time")
ax1.legend(loc="best")
ax1.grid(alpha=0.3)

gaps = [abs(h[3]) for h in game.history]
ax2.bar(rounds, gaps, color=["#2a9d8f" if g <= game.tolerance else "#e76f51" for g in gaps])
ax2.axhline(game.tolerance, ls="--", color="gray", lw=1)
ax2.set_ylabel("|AS - AD|")
ax2.set_xlabel("Round")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Interactive version (ipywidgets)

Run this cell inside Jupyter to play it yourself. Pick a role, start a round to
draw a shock, then use your levers to bring the economy back into balance.
Watch your **budget points** and **technology tokens** — when the budget runs
out, the game ends and your **score** is the number of rounds you kept the
economy balanced.

> On GitHub the widgets render as static controls; clone and run locally for the
> full interactive experience.

In [4]:
import ipywidgets as widgets
from IPython.display import display

game = EconBalancer(role="Economist", seed=None)

# --- display widgets ---
role_dd      = widgets.Dropdown(options=list(ROLES), value="Economist", description="Role:")
scenario_lbl = widgets.HTML(value="<b>Press 'New Round' to begin.</b>")
stats_lbl    = widgets.HTML()
result_lbl   = widgets.HTML()
tech_chk     = widgets.Checkbox(value=False, description="Use a technology token (2x effect)")

new_round_btn = widgets.Button(description="New Round", button_style="info")
sup_up   = widgets.Button(description="↑ Supply")
sup_dn   = widgets.Button(description="↓ Supply")
dem_up   = widgets.Button(description="↑ Demand")
dem_dn   = widgets.Button(description="↓ Demand")
reset_btn = widgets.Button(description="Reset", button_style="warning")

action_btns = [sup_up, sup_dn, dem_up, dem_dn]

def refresh():
    stats_lbl.value = (
        f"Round <b>{game.round_no}</b> &nbsp;|&nbsp; "
        f"AS <b>{game.AS}</b> &nbsp; AD <b>{game.AD}</b> &nbsp; gap <b>{game.gap}</b><br>"
        f"Budget <b>{game.budget_points}</b> &nbsp;|&nbsp; "
        f"Tech tokens <b>{game.technology_tokens}</b> &nbsp;|&nbsp; "
        f"Score <b>{game.score}</b>"
    )
    tech_chk.disabled = game.technology_tokens == 0
    out_of_budget = not game.can_act()
    for b in action_btns:
        b.disabled = out_of_budget or game.current_scenario is None
    if out_of_budget:
        result_lbl.value = f"<b>Game over!</b> Final score: {game.score} rounds in equilibrium."

def on_new_round(_):
    name, ds, dd = game.new_round()
    scenario_lbl.value = f"<b>Shock:</b> {name} (AS {ds:+}, AD {dd:+})"
    result_lbl.value = game.status()
    refresh()

def make_action(lever, direction):
    def handler(_):
        if game.current_scenario is None:
            result_lbl.value = "Press 'New Round' first."
            return
        msg = game.apply_policy(lever, direction, use_tech=tech_chk.value)
        tech_chk.value = False
        result_lbl.value = msg
        refresh()
    return handler

def on_role_change(change):
    game.role = change["new"]
    refresh()

def on_reset(_):
    global game
    game = EconBalancer(role=role_dd.value, seed=None)
    scenario_lbl.value = "<b>Press 'New Round' to begin.</b>"
    result_lbl.value = ""
    refresh()

new_round_btn.on_click(on_new_round)
sup_up.on_click(make_action("supply", +1))
sup_dn.on_click(make_action("supply", -1))
dem_up.on_click(make_action("demand", +1))
dem_dn.on_click(make_action("demand", -1))
reset_btn.on_click(on_reset)
role_dd.observe(on_role_change, names="value")

refresh()
display(widgets.VBox([
    widgets.HBox([role_dd, new_round_btn, reset_btn]),
    scenario_lbl,
    stats_lbl,
    widgets.HBox([sup_up, sup_dn, dem_up, dem_dn]),
    tech_chk,
    result_lbl,
]))
